---
title: Query via h5netcdf
---

# Query NISAR GUNW via h5netcdf (baseline)

For comparison, open the same NISAR granule the traditional way using
earthaccess and h5netcdf, and extract the same spatial subset.

In [ ]:
import warnings

warnings.filterwarnings("ignore", category=FutureWarning, module="earthaccess")

## Open with earthaccess + h5netcdf

In [ ]:
import time

import earthaccess
import xarray as xr

earthaccess.login()

results = earthaccess.search_data(
    short_name="NISAR_L2_GUNW_BETA_V1",
    point=(174.1192, -39.3379),  # lon, lat
    temporal=("2026-01-01", "2026-01-04"),
)

files = earthaccess.open(results)
t0 = time.perf_counter()
ds = xr.open_dataset(
    files[0],
    group="/science/LSAR/GUNW/grids/frequencyA/unwrappedInterferogram/HH",
    engine="h5netcdf",
    decode_timedelta=False,
    phony_dims="access",
)
print(f"Opened in {time.perf_counter() - t0:.2f}s")
ds

## Extract the same spatial subset

In [ ]:
import numpy as np

ny, nx = ds.dims["yCoordinates"], ds.dims["xCoordinates"]

# Same 500x500 pixel window from the center of the scene
cy, cx = ny // 2, nx // 2
subset_slice = dict(
    yCoordinates=slice(cy - 250, cy + 250), xCoordinates=slice(cx - 250, cx + 250)
)

t0 = time.perf_counter()
subset = ds.isel(**subset_slice).load()
elapsed = time.perf_counter() - t0

print(f"Loaded 500x500 subset in {elapsed:.2f}s (from {ny}x{nx} full scene)")
subset

## Visualize the subset

In [ ]:
import pygmt

fig = pygmt.Figure()
pygmt.makecpt(
    cmap="SCM/romaO", series=[0, 2 * np.pi, np.pi / 2], cyclic=True, continuous=True
)
fig.grdimage(grid=subset.unwrappedPhase, cmap=True, frame=True)
fig.colorbar()
fig.show()

## Visualize coherence magnitude

In [ ]:
fig = pygmt.Figure()
fig.grdimage(grid=subset.coherenceMagnitude, cmap="gray", frame=True)
fig.colorbar()
fig.show()

Note: `earthaccess.open` streams the file through HTTPS via fsspec.
Loading a spatial subset may still transfer a large portion of the file
depending on the HDF5 chunking layout. Compare the wall-clock time
with the Icechunk notebook, which fetches only the chunks overlapping
the subset.